In [ ]:
# run this notebook after 19_distribution_genome_figure_files_R
# use this notebook to generate heatmap distribution of sibling differences in 1 mb windows across the genome,
# c>a rate by replication timing, cpg>tpg rate by methylation, ukb and aou sibling pair #dnm histograms
# after this, run all notebooks in ../pca/

In [ ]:
library(jcolors)
library(data.table)
library(ggplot2)
library(dplyr)
library(jsonlite)
library(tidyverse)
library(ggpubr)
library(purrr)
library(tidyr)
library(emmeans)
library(RColorBrewer)
library(GenomicRanges)
library(rtracklayer)
library(grid)
library(cowplot)
library(bedtoolsr)

In [ ]:
aou_difs = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt")
ukb_difs = fread("./ukb/ukb_sibs_dnms_all.csv", sep = ",")
giab = fread("./data/giab_difficult_merged.bed")
names(giab) = c("chr", "start", "end", "length")
aou_difs = aou_difs %>% select("chr", "pos")
aou_difs$dataset = "aou"
names(ukb_difs) = c("chr", "pos")
ukb_difs$dataset = "ukb"
dnms = rbind(aou_difs, ukb_difs)
dnms$chr = paste0("chr", dnms$chr)
giab$chr = paste0("chr", giab$chr)

In [ ]:
# genome distribution heatmap 

# ── parameters ────────────────────────────────────────────────────────────────
BIN_SIZE  = 1e6
CHR_ORDER = paste0("chr", c(1:22))

CHR_SIZES = c(
  chr1=248956422, chr2=242193529, chr3=198295559, chr4=190214555,
  chr5=181538259, chr6=170805979, chr7=159345973, chr8=145138636,
  chr9=138394717, chr10=133797422, chr11=135086622, chr12=133275309,
  chr13=114364328, chr14=107043718, chr15=101991189, chr16=90338345,
  chr17=83257441,  chr18=80373285,  chr19=58617616,  chr20=64444167,
  chr21=46709983,  chr22=50818468
)
N_SAMPLES = c(ukb = 21660, aou = 7325)


# ── global theme ──────────────────────────────────────────────────────────────
base_theme = theme_classic(base_size = 10) +
  theme(
    axis.text        = element_text(size = 8),
    axis.title       = element_text(size = 9),
    legend.text      = element_text(size = 8),
    legend.title     = element_text(size = 9),
    strip.text       = element_text(size = 9),
    plot.title       = element_text(size = 10, face = "bold"),
    axis.ticks.length = unit(2, "pt"),
    panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5)
  )

theme_set(base_theme)


In [ ]:
# ── 1. bin dnms and compute density ───────────────────────────────────────────
dnms_binned = dnms |>
  filter(chr %in% CHR_ORDER) |>
  mutate(
    chr = factor(chr, levels = CHR_ORDER),
    bin = floor(pos / BIN_SIZE) * BIN_SIZE
  ) |>
  count(chr, dataset, bin, name = "n_dnm") |>
  mutate(density = n_dnm / N_SAMPLES[dataset] * 1000)

# ── 2. full bin grid ──────────────────────────────────────────────────────────
plot_df = tibble(chr = CHR_ORDER, size = CHR_SIZES[CHR_ORDER]) |>
  mutate(chr = factor(chr, levels = CHR_ORDER)) |>
  rowwise() |>
  reframe(chr = chr, bin = seq(0, size - 1, by = BIN_SIZE)) |>
  crossing(dataset = c("ukb", "aou")) |>
  left_join(dnms_binned, by = c("chr", "dataset", "bin")) |>
  mutate(
    n_dnm   = replace_na(n_dnm, 0L),
    density = replace_na(density, 0)
  )

# ── 2b. build bins granges (shared for both flag steps) ──────────────────────
bins_distinct = plot_df |> distinct(chr, bin)

bins_gr = makeGRangesFromDataFrame(
  bins_distinct |> mutate(start = bin, end = bin + BIN_SIZE - 1),
  seqnames.field = "chr", start.field = "start", end.field = "end"
)

# ── 2c. flag bins with >50% giab coverage ────────────────────────────────────
giab_gr = makeGRangesFromDataFrame(giab, seqnames.field = "chr")
seqlevels(giab_gr, pruning.mode = "coarse") = seqlevels(bins_gr)

covered_bp_giab = findOverlaps(bins_gr, giab_gr) |>
  (\(hits) tibble(
    idx        = queryHits(hits),
    overlap_bp = width(pintersect(bins_gr[queryHits(hits)], giab_gr[subjectHits(hits)]))
  ))() |>
  group_by(idx) |>
  summarise(covered_bp = sum(overlap_bp), .groups = "drop")

giab_flags = bins_distinct |>
  mutate(idx = row_number()) |>
  left_join(covered_bp_giab, by = "idx") |>
  mutate(
    covered_bp = replace_na(covered_bp, 0),
    is_giab    = covered_bp / BIN_SIZE > 0.5
  ) |>
  select(chr, bin, is_giab)

# ── 2d. flag bins overlapping hg38 assembly gaps ─────────────────────────────
gaps_gr = fread(
  "./data/hg38_gaps_true.txt", header=TRUE) %>%
  select(chrom, chromStart, chromEnd) |>
  makeGRangesFromDataFrame(
    seqnames.field = "chrom",
    start.field    = "chromStart",
    end.field      = "chromEnd"
  )
seqlevels(gaps_gr, pruning.mode = "coarse") = seqlevels(bins_gr)

covered_bp_gap = findOverlaps(bins_gr, gaps_gr) |>
  (\(hits) tibble(
    idx        = queryHits(hits),
    overlap_bp = width(pintersect(bins_gr[queryHits(hits)], gaps_gr[subjectHits(hits)]))
  ))() |>
  group_by(idx) |>
  summarise(covered_bp = sum(overlap_bp), .groups = "drop")

gap_flags = bins_distinct |>
  mutate(idx = row_number()) |>
  left_join(covered_bp_gap, by = "idx") |>
  mutate(
    covered_bp = replace_na(covered_bp, 0),
    is_gap     = covered_bp / BIN_SIZE > 0.001
  ) |>
  select(chr, bin, is_gap)

# ── 2e. join flags onto plot_df ───────────────────────────────────────────────
plot_df = plot_df |>
  left_join(giab_flags, by = c("chr", "bin")) |>
  left_join(gap_flags,  by = c("chr", "bin")) |>
  mutate(
    is_giab = replace_na(is_giab, FALSE),
    is_gap  = replace_na(is_gap,  FALSE)
  )

# ── 3. map density to colors ──────────────────────────────────────────────────
reds  = colorRampPalette(brewer.pal(9, "Reds"))(100)
blues = colorRampPalette(brewer.pal(9, "Blues"))(100)

max_density = max(plot_df$density[!plot_df$is_giab & !plot_df$is_gap], na.rm = TRUE)

n_chr = length(CHR_ORDER)

# geometry: tile_height = 2 * sub_y_offset so the two tiles meet flush;
# border_half slightly larger to enclose both tiles with a little padding.
sub_y_offset = 0.165
tile_height  = 0.33
border_half  = 0.34

plot_df = plot_df |>
  mutate(
    fill_color = case_when(
      is_gap           ~ "#808080",
      is_giab          ~ "#c0bdb5",
      dataset == "ukb" ~ reds[pmax(1, ceiling(density / max_density * 100))],
      dataset == "aou" ~ blues[pmax(1, ceiling(density / max_density * 100))]
    ),
    chr_y  = n_chr + 1 - as.integer(chr),
    sub_y  = chr_y + if_else(dataset == "ukb", sub_y_offset, -sub_y_offset),
    bin_mb = bin / 1e6
  )

# ── 4. chr borders and dataset labels ────────────────────────────────────────
chr_borders = plot_df |>
  group_by(chr, chr_y) |>
  summarise(x_max = max(bin_mb) + 1, .groups = "drop")

# ── 5. main plot ──────────────────────────────────────────────────────────────
p = ggplot(plot_df, aes(x = bin_mb + 0.5, y = sub_y)) +
  geom_tile(
    aes(fill = I(fill_color), width = 1, height = tile_height),
    color = NA
  ) +
  geom_rect(
    data        = chr_borders,
    aes(xmin = 0, xmax = x_max,
        ymin = chr_y - border_half, ymax = chr_y + border_half),
    fill        = NA,
    color       = "black",
    linewidth   = 0.3,
    inherit.aes = FALSE
  ) +
  scale_x_continuous(name = "Position (Mb)", expand = c(0, 2)) +
  scale_y_continuous(
    breaks = n_chr:1,
    labels = sub("chr", "", CHR_ORDER),
    expand = c(0.01, 0)
  ) +
  coord_cartesian(clip = "off") +
  labs(y = "Chromosome") +
  theme_classic(base_size = 14) +
  theme(
    axis.text.y       = element_text(size = 10),
    axis.text.x       = element_text(size = 10),
    axis.ticks.length = unit(2, "pt"),
    panel.border      = element_rect(color = "black", fill = NA, linewidth = 0.5),
    plot.margin       = margin(5, 5, 5, 5),
    legend.position   = "none"
  )

# ── 6. color bar legends ──────────────────────────────────────────────────────
tick_vals = pretty(c(0, max_density), n = 4)
tick_vals = tick_vals[tick_vals <= max_density]

make_colorbar = function(colors, title, tick_vals, max_val) {
  n       = length(colors)
  bar_df  = tibble(y = seq(0, 1, length.out = n), fill = colors)
  tick_df = tibble(y = tick_vals / max_val, label = as.character(tick_vals))
  ggplot() +
    geom_tile(data = bar_df,
              aes(x = 0.5, y = y, fill = I(fill), height = 1/n, width = 1)) +
    geom_segment(data = tick_df,
                 aes(x = 1, xend = 1.2, y = y, yend = y),
                 linewidth = 0.3, color = "black") +
    geom_text(data = tick_df,
              aes(x = 1.3, y = y, label = label),
              size = 4.5, hjust = 0, color = "black") +
    annotate("text", x = 0.5, y = 1.1, label = title,
             size = 5, hjust = 0.5, color = "black") +
    annotate("rect", xmin = 0, xmax = 1, ymin = 0, ymax = 1,
             fill = NA, color = "black", linewidth = 0.3) +
    scale_x_continuous(limits = c(0, 3.5)) +
    scale_y_continuous(limits = c(-0.05, 1.2)) +
    theme_void() +
    theme(plot.margin = margin(0, 0, 0, 0))
}

cb_ukb = make_colorbar(reds,  "UKB", tick_vals, max_density)
cb_aou = make_colorbar(blues, "AoU", tick_vals, max_density)

giab_legend = ggplot() +
  annotate("rect", xmin=0, xmax=1, ymin=0.55, ymax=0.95,
           fill="#c0bdb5", color="black", linewidth=0.3) +
  annotate("text", x=1.2, y=0.75, label="GIAB >50%",
           size=5, hjust=0, color="black") +
  annotate("rect", xmin=0, xmax=1, ymin=0.05, ymax=0.45,
           fill="#808080", color="black", linewidth=0.3) +
  annotate("text", x=1.2, y=0.25, label="hg38 gap",
           size=5, hjust=0, color="black") +
  scale_x_continuous(limits=c(0, 5)) +
  scale_y_continuous(limits=c(0, 1.1)) +
  theme_void()

units_label = ggplot() +
  annotate("text", x = 0.5, y = 0.5, label = "DNMs per Mb\nper 1,000 sibling pairs",
           size = 4.5, hjust = 0.5, color = "black") +
  scale_x_continuous(limits = c(0, 1)) +
  scale_y_continuous(limits = c(0, 1)) +
  theme_void()

legend_panel = plot_grid(
  units_label, cb_ukb, cb_aou, giab_legend,
  ncol = 1, rel_heights = c(0.8, 3, 3, 1.2)
)

# ── 7. combine and save ───────────────────────────────────────────────────────
final = plot_grid(p, legend_panel, ncol = 2, rel_widths = c(10, 1.6))


In [ ]:
# chr 8 inset 

aou_obs = 0.222
ukb_obs = 0.219
n_pairs = c("aou" = 7325*aou_obs*4, "ukb" = 21660*ukb_obs*4)
windows = fread("./chr8_cg_opportunity.tsv")
windows = windows %>% mutate(dnm_opportunity = end-start - num_bases_giab,
    aou_cg_rate = n_aou_cg/n_pairs["aou"]/cg_opportunity,
                             aou_ca_rate = n_aou_ca/n_pairs["aou"]/cg_opportunity,
    aou_rate = n_aou/n_pairs["aou"]/dnm_opportunity,
                            ukb_cg_rate = n_ukb_cg/n_pairs["ukb"]/cg_opportunity,
                             ukb_ca_rate = n_ukb_ca/n_pairs["ukb"]/cg_opportunity,
                            ukb_rate = n_ukb/n_pairs["ukb"]/dnm_opportunity)
windows = windows %>% filter(start != 7e6 & end <= 10e6)

df = windows

df_long = df |>
  mutate(window = start / 1e6) |>
  select(window,
         aou_cg_rate, aou_ca_rate, aou_rate,
         ukb_cg_rate, ukb_ca_rate, ukb_rate) |>
  pivot_longer(
    cols      = -window,
    names_to  = "series",
    values_to = "rate"
  ) |>
  mutate(
    cohort   = if_else(startsWith(series, "aou"), "AoU", "UKB"),
    type     = case_when(
      grepl("_cg_", series) ~ "C>G rate",
      grepl("_ca_", series) ~ "C>A rate",
    ),
    linetype = case_when(
      type == "C>G rate"     ~ "solid",
      type == "C>A rate"     ~ "dotted"
    )
  )

df_long= df_long %>% filter(!is.na(type))
ggplot(df_long, aes(x = window, y = rate,
                    color = cohort, linetype = type,
                    group = series)) +
  annotate("rect",
           xmin = 6.5, xmax = 7.5,
           ymin = -Inf, ymax = Inf,
           fill = "grey80", alpha = 0.5) +
  annotate("text",
           x = 7, y = Inf,
           label = "GIAB\ndifficult",
           vjust = 1.3, size = 3, color = "grey40") +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  scale_color_manual(values = c("AoU" = "#1a5f8a", "UKB" = "#b03a2e")) +
  scale_linetype_manual(values = c(
    "C>G rate"     = "solid",
    "C>A rate"     = "dotted"
  )) +
  scale_y_continuous(labels = scales::scientific) +
  scale_x_continuous(
    breaks = df$start / 1e6,
    labels = paste0(df$start / 1e6, "-", df$end / 1e6," Mb")
  ) +
  labs(
    x        = "Chromosome 8 position (Mb)",
    y        = "DNM rate (per bp)",
    color    = "Cohort",
    linetype = "Rate type"
  ) +
  theme_classic() +
  theme(
    axis.text.x      = element_text(angle = 45, hjust = 1, size = 8),
    legend.position  = "bottom",
    legend.key.width = unit(1.5, "cm")
  )

In [ ]:
# cpg by methylation 
meth_counts = c(5146964, 1707806, 3986837, 8049673, 30458069)

n_pairs = c("aou" = 7325*aou_obs*4, "ukb" = 21660*ukb_obs*4)

plot_data = fread("./cpg_with_meth.tsv") %>%
  filter(!is.na(methylation)) %>%
  mutate(methylation = pmin(methylation, 100),
         meth_bin = cut(methylation, breaks = seq(0, 100, 20),
                        labels = paste0(seq(0, 80, 20), "-", seq(20, 100, 20), "%"),
                        include.lowest = TRUE, right = TRUE)) %>%
  group_by(dataset, meth_bin) %>%
  summarise(n_muts = n(), .groups = "drop") %>%
  mutate(
    muts_per_pair = n_muts / n_pairs[dataset],
    label = paste0(scales::comma(n_muts))
  )

plot_data$muts_rate = plot_data$muts_per_pair/c(meth_counts, meth_counts)

p_methylation = plot_data %>%
  ggplot(aes(x = meth_bin, y = muts_rate, color = dataset, group = dataset)) +
  geom_point(size = 3) + geom_line(linewidth = 1.2) + 
  scale_color_manual(values = c("aou" = "steelblue", "ukb" = "firebrick")) +
  labs(x = "Testis CpG Methylation Level", y = "CpG>TpG Mutation Rate", color = "Dataset") +
  theme_bw() + scale_y_continuous(
    labels = scales::scientific,
    breaks = scales::pretty_breaks(n = 6)
  ) + 
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) 
p_methylation

In [ ]:
# c to a by replication timing 

rt = fread("./rt_windows.bed")
names(rt)[2:3] = c("start", "end")
rt$chr = paste0("chr", rt$chr)
opp = fread("./rt_windows_ca_opportunity_giab_masked.tsv")
rt = merge(rt, opp, by = c("chr", "start", "end"))
rt_plot = data.frame(rank = 1:10, n_aou = NA, n_ukb = NA, n_ca_opp = NA, rt = NA)
aou_bed = data.frame(chrom = paste0("chr", aou_ca$chr), start = aou_ca$pos, end = aou_ca$pos + 1)
ukb_bed = data.frame(chrom = paste0("chr", ukb_ca$chr), start = ukb_ca$pos, end = ukb_ca$pos + 1)
for (i in 1:nrow(rt_plot)){
    sub = rt %>% filter(rt_rank == i)
    sub_bed = data.frame(chrom = sub$chr, start = sub$start, end = sub$end)
    merged_aou = bedtoolsr::bt.intersect(sub_bed, aou_bed, c=TRUE)
    merged_ukb = bedtoolsr::bt.intersect(sub_bed, ukb_bed, c=TRUE)
    rt_plot$n_ca_opp[i] = sum(sub$ca_opportunity)
    rt_plot$n_aou[i] = sum(merged_aou$V4[which(merged_aou$V4 > 0)])
    rt_plot$n_ukb[i] = sum(merged_ukb$V4[which(merged_ukb$V4 > 0)])
    rt_plot$rt[i] = mean(sub$mean_rt)
}
n_pairs = c("aou" = 7325*aou_obs*4, "ukb" = 21660*ukb_obs*4)

rt_plot = rt_plot %>% mutate(aou_rate = n_aou/n_ca_opp/n_pairs["aou"],
                            ukb_rate = n_ukb/n_ca_opp/n_pairs["ukb"])
rt_plot$rank = 11-rt_plot$rank

library(ggplot2)
library(dplyr)

p_rt = rt_plot %>%
  select(rank, aou_rate, ukb_rate) %>%
  pivot_longer(cols = c(aou_rate, ukb_rate),
               names_to = "dataset",
               values_to = "rate") %>%
  mutate(dataset = recode(dataset,
                          "aou_rate" = "aou",
                          "ukb_rate" = "ukb")) %>%
  ggplot(aes(x = rank, y = rate, color = dataset, group = dataset)) +
  geom_line(linewidth=1.2) +
  geom_point(size = 3) +
  scale_color_manual(values = c("aou" =  "#1a5f8a", "ukb" = "#b03a2e")) +
  scale_x_continuous(breaks = 1:10) +
  labs(x = "Replication timing bin (1 = earliest, 10 = latest)",
       y = "C>A Mutation Rate",
       color = "Dataset") + scale_y_continuous(
    labels = scales::scientific,
        breaks = seq(min(rt_plot$aou_rate, rt_plot$ukb_rate),
               max(rt_plot$aou_rate, rt_plot$ukb_rate),
               length.out = 5)  ) + 
  theme_bw()
p_rt